# 03 GenAI Benchmark Best Model

Purpose: run the selected OpenAI GPT-4.1-mini labeling workflow and evaluate it against the locked human consensus holdout. This notebook is sanitized for GitHub: it does not include API keys or local machine paths.


## Setup

Set `OPENAI_API_KEY` in the notebook environment before running. In Colab, use Secrets or an environment variable. Do not paste keys into this notebook.


In [ ]:
from pathlib import Path
import os

BASE = Path.cwd()
if not (BASE / 'scripts' / 'genai_benchmark.py').exists():
    # In Colab, update this path to your Drive repo folder after mounting Drive.
    BASE = Path('/content/drive/MyDrive/goodreads-reader-response-classifier')

os.chdir(BASE)
print('Working directory:', Path.cwd())
assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY before running benchmark cells.'


## Benchmark GPT-4.1-mini on Locked Holdout

This produces GPT-4.1-mini labels for the locked holdout and evaluates them against human consensus labels.


In [ ]:
!python scripts/genai_benchmark.py   --supplier OpenAI   --model gpt-4.1-mini   --input data/processed/holdout_locked.csv   --output data/results/holdout_openai_gpt-4.1-mini.csv


In [ ]:
!python scripts/evaluate_predictions.py   --gold data/results/holdout_human_consensus.csv   --pred data/results/holdout_openai_gpt-4.1-mini.csv   --output data/results/holdout_openai_gpt-4.1-mini_eval.json


## Label Train/Test With GPT-4.1-mini

Run these cells only after the holdout benchmark is complete and the team has confirmed GPT-4.1-mini is the labeling model for this run.


In [ ]:
!python scripts/genai_benchmark.py   --supplier OpenAI   --model gpt-4.1-mini   --input data/processed/train_no_holdout_overlap.csv   --output data/results/train_labeled.csv


In [ ]:
!python scripts/genai_benchmark.py   --supplier OpenAI   --model gpt-4.1-mini   --input data/processed/test_no_holdout_overlap.csv   --output data/results/test_labeled.csv


## Quick Output Checks


In [ ]:
import json
from pathlib import Path
import pandas as pd

for path in [
    Path('data/results/holdout_openai_gpt-4.1-mini.csv'),
    Path('data/results/holdout_openai_gpt-4.1-mini_eval.json'),
    Path('data/results/train_labeled.csv'),
    Path('data/results/test_labeled.csv'),
]:
    print(path, 'exists:', path.exists())

if Path('data/results/holdout_openai_gpt-4.1-mini_eval.json').exists():
    print(json.dumps(json.loads(Path('data/results/holdout_openai_gpt-4.1-mini_eval.json').read_text()), indent=2)[:2000])

if Path('data/results/holdout_openai_gpt-4.1-mini.csv').exists():
    display(pd.read_csv('data/results/holdout_openai_gpt-4.1-mini.csv').head())
